# Psychology-Informed Anomaly Detection in Insider Threat Systems
## Google Colab Notebook

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim

import random
from google.colab import files

## 2. Set Random Seed

In [ ]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

## 3. Upload and Load `logon.csv`

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df.head()

## 4. Check Required Columns

In [ ]:
df.columns = df.columns.str.lower().str.strip()

required_columns = {"date", "user", "pc", "activity"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df.head()

## 5. Convert Timestamp into Time Fields

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(subset=["date", "user", "pc", "activity"])

df["day"] = df["date"].dt.date
df["hour"] = df["date"].dt.hour
df["weekday"] = df["date"].dt.weekday

df[["date", "day", "hour", "weekday", "user", "pc", "activity"]].head()

## 6. Create Behavioural Indicators

In [ ]:
df["activity_clean"] = df["activity"].astype(str).str.lower().str.strip()

df["is_logon"] = (df["activity_clean"] == "logon").astype(int)
df["is_logoff"] = (df["activity_clean"] == "logoff").astype(int)

df["weekend"] = (df["weekday"] >= 5).astype(int)

WORK_START = 8
WORK_END = 18

df["after_hours"] = ((df["hour"] < WORK_START) | (df["hour"] >= WORK_END)).astype(int)

df[["activity", "is_logon", "is_logoff", "weekend", "after_hours"]].head()

## 7. Aggregate Events into User-Day Profiles

In [ ]:
features = df.groupby(["user", "day"], as_index=False).agg(
    total_events=("activity", "count"),
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    unique_hosts=("pc", "nunique"),
    first_activity_hour=("hour", "min"),
    last_activity_hour=("hour", "max"),
    after_hours_count=("after_hours", "sum"),
    weekend_flag=("weekend", "max"),
    hour_std=("hour", "std")
)

features["hour_std"] = features["hour_std"].fillna(0)

features.head()

## 8. Create First Login Hour

In [ ]:
first_login = (
    df[df["is_logon"] == 1]
    .groupby(["user", "day"])["hour"]
    .min()
    .reset_index()
    .rename(columns={"hour": "first_login_hour"})
)

features = features.merge(first_login, on=["user", "day"], how="left")

features["first_login_hour"] = features["first_login_hour"].fillna(features["first_activity_hour"])

features = features.drop(columns=["first_activity_hour"])

features.head()

## 9. Create Ratio Features

In [ ]:
features["after_hours_ratio"] = features["after_hours_count"] / features["total_events"]

features["logon_logoff_ratio"] = features["logon_count"] / (features["logoff_count"] + 1)

features = features.replace([np.inf, -np.inf], np.nan)
features = features.dropna()

features.head()

## 10. Select Model Input Features

In [ ]:
feature_cols = [
    "total_events",
    "logon_count",
    "logoff_count",
    "unique_hosts",
    "first_login_hour",
    "last_activity_hour",
    "after_hours_count",
    "weekend_flag",
    "hour_std",
    "after_hours_ratio",
    "logon_logoff_ratio"
]

X_raw = features[feature_cols].copy()

X_raw.head()

## 11. Chronological Train-Test Split

In [ ]:
features["day"] = pd.to_datetime(features["day"])

all_days = sorted(features["day"].unique())

split_position = int(len(all_days) * 0.8)
split_day = all_days[split_position]

train_data = features[features["day"] < split_day].copy()
test_data = features[features["day"] >= split_day].copy()

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))
print("Split day:", split_day)

## 12. Scale Features

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train_data[feature_cols])
X_test = scaler.transform(test_data[feature_cols])

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

## 13. Convert Data to PyTorch Tensors

In [ ]:
X_train_tensor = torch.tensor(X_train)
X_test_tensor = torch.tensor(X_test)

## 14. Build Dense Autoencoder

In [ ]:
class DenseAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(DenseAutoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


input_dim = X_train.shape[1]
model = DenseAutoencoder(input_dim)

model

## 15. Train Autoencoder

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50

for epoch in range(epochs):
    model.train()
    
    optimizer.zero_grad()
    
    reconstructed = model(X_train_tensor)
    loss = criterion(reconstructed, X_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.6f}")

## 16. Calculate Reconstruction Errors

In [ ]:
model.eval()

with torch.no_grad():
    train_reconstructed = model(X_train_tensor)
    test_reconstructed = model(X_test_tensor)

train_errors = torch.mean((X_train_tensor - train_reconstructed) ** 2, dim=1).numpy()
test_errors = torch.mean((X_test_tensor - test_reconstructed) ** 2, dim=1).numpy()

print("First 5 test anomaly scores:")
print(test_errors[:5])

## 17. Calculate Anomaly Threshold

In [ ]:
threshold = train_errors.mean() + 2 * train_errors.std()

print("Mean training error:", train_errors.mean())
print("Training error standard deviation:", train_errors.std())
print("Anomaly threshold:", threshold)

## 18. Create Anomaly Results Table

In [ ]:
results = test_data[["user", "day"] + feature_cols].copy()

results["anomaly_score"] = test_errors
results["anomaly_label"] = (results["anomaly_score"] > threshold).astype(int)

results.head()

## 19. Calculate Feature-Level Reconstruction Errors

In [ ]:
test_feature_errors = (X_test_tensor.numpy() - test_reconstructed.numpy()) ** 2

feature_error_df = pd.DataFrame(
    test_feature_errors,
    columns=feature_cols,
    index=results.index
)

results["top_features"] = feature_error_df.apply(
    lambda row: list(row.sort_values(ascending=False).head(3).index),
    axis=1
)

results[["user", "day", "anomaly_score", "anomaly_label", "top_features"]].head()

## 20. Create 95th Percentile Feature Cutoffs

In [ ]:
p95_features = train_data[feature_cols].quantile(0.95)
p05_features = train_data[feature_cols].quantile(0.05)

score_p95_train = np.percentile(train_errors, 95)

print("95th percentile feature cutoffs:")
print(p95_features)

print("\n5th percentile feature cutoffs:")
print(p05_features)

print("\n95th percentile training anomaly score:")
print(score_p95_train)

## 21. Create Severity and Persistence Cutoffs

In [ ]:
anomalous_results = results[results["anomaly_label"] == 1].copy()

if len(anomalous_results) > 0:
    severe_cutoff = anomalous_results["anomaly_score"].quantile(0.95)
else:
    severe_cutoff = np.inf

user_anomaly_counts = anomalous_results["user"].value_counts()

if len(user_anomaly_counts) > 0:
    persistence_cutoff = user_anomaly_counts.quantile(0.95)
else:
    persistence_cutoff = np.inf

print("Severe deviation cutoff:", severe_cutoff)
print("Persistence cutoff:", persistence_cutoff)

## 22. Assign Psychology-Informed Categories

In [ ]:
boundary_features = ["after_hours_ratio", "after_hours_count", "unique_hosts", "weekend_flag"]
volatility_features = ["hour_std", "last_activity_hour", "first_login_hour"]
spike_features = ["total_events", "logon_count", "logoff_count", "unique_hosts"]


def has_top_feature(top_features, candidate_features):
    return any(feature in top_features for feature in candidate_features)


def assign_category(row):
    if row["anomaly_label"] == 0:
        return "Normal"
    
    categories = []
    
    boundary_condition = (
        (row["after_hours_ratio"] > p95_features["after_hours_ratio"]) or
        (row["after_hours_count"] > p95_features["after_hours_count"]) or
        (row["unique_hosts"] > p95_features["unique_hosts"]) or
        (row["weekend_flag"] > p95_features["weekend_flag"])
    )
    
    if boundary_condition and has_top_feature(row["top_features"], boundary_features):
        categories.append("Boundary Crossing")
    
    volatility_condition = (
        (row["hour_std"] > p95_features["hour_std"]) or
        (row["last_activity_hour"] > p95_features["last_activity_hour"]) or
        (row["first_login_hour"] < p05_features["first_login_hour"])
    )
    
    if volatility_condition and has_top_feature(row["top_features"], volatility_features):
        categories.append("Behavioural Volatility")
    
    spike_condition = (
        (row["anomaly_score"] > score_p95_train) and
        (row["total_events"] > p95_features["total_events"])
    )
    
    if spike_condition and has_top_feature(row["top_features"], spike_features):
        categories.append("Impulsive Spike")
    
    if row["anomaly_score"] > severe_cutoff:
        categories.append("Severe Deviation")
    
    if user_anomaly_counts.get(row["user"], 0) > persistence_cutoff:
        categories.append("Persistence")
    
    if len(categories) == 0:
        categories.append("General Anomaly")
    
    return " | ".join(categories)


results["psychology_category"] = results.apply(assign_category, axis=1)

results[["user", "day", "anomaly_score", "anomaly_label", "top_features", "psychology_category"]].head(10)

## 23. View Highest Scoring Anomalies

In [ ]:
final_results = results[
    [
        "user",
        "day",
        "anomaly_score",
        "anomaly_label",
        "top_features",
        "psychology_category"
    ] + feature_cols
].copy()

final_results.sort_values("anomaly_score", ascending=False).head(20)

## 24. Category Summary

In [ ]:
category_summary = final_results["psychology_category"].value_counts().reset_index()
category_summary.columns = ["psychology_category", "count"]

category_summary

## 25. Save Final Results

In [ ]:
final_results.to_csv("final_anomaly_results.csv", index=False)

files.download("final_anomaly_results.csv")

## 26. Save Model

In [ ]:
torch.save(model.state_dict(), "dense_autoencoder_model.pth")

files.download("dense_autoencoder_model.pth")